# Topology-Aware Dynamics (TAD) — E1 on a Colab GPU

Runs the spec's **E1 minimal-linear-crux** end to end on a Colab GPU:
generate trajectories → build dynamics dataset → train/evaluate predictors and
strong baselines → oracle candidate selection + predicted-subspace optimizer →
first report.

**Before running:** `Runtime → Change runtime type → Hardware accelerator = GPU`.
`REPO_URL` is preset to `github.com/rtavasso/topologyOptimizer`; edit it only if
you forked the repo (or upload the repo folder and skip the clone cell).

In [ ]:
# 1. GPU check
!nvidia-smi || echo 'No GPU — enable it via Runtime > Change runtime type > GPU'

In [ ]:
# 2. Parameters
REPO_URL = 'https://github.com/rtavasso/topologyOptimizer.git'
REPO_DIR = 'topologyOptimizer'
CONFIG   = 'configs/experiments/e1_colab.yaml'   # session-sized E1; use configs/experiments/e1.yaml for the full run
DEVICE   = 'cuda'

In [ ]:
# 3. Locate or clone the repository, then cd into it
import os
if os.path.isfile('pyproject.toml') and os.path.isdir('src/tad'):
    print('Already in the repo root:', os.getcwd())
elif os.path.isdir(REPO_DIR):
    %cd {REPO_DIR}
else:
    !git clone {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}
print('cwd =', os.getcwd())
assert os.path.isfile('pyproject.toml'), 'Run from the repository root (set REPO_URL or upload the repo).'

In [ ]:
# 4. Install dependencies + TAD (does not touch the preinstalled CUDA torch)
!bash scripts/colab_setup.sh

## Kick off the experiment

`run-e1` runs every stage in one process and streams progress. The model
forward/backward and the learned predictors run on the GPU; dense per-step
logging (SVDs, covariances, chunked Zarr writes) is CPU/IO-bound by design
(“discovery mode”, spec §8.1).

In [ ]:
# 5. Run the full E1 pipeline on the GPU
#    (--device is a top-level flag, so it precedes the subcommand)
!python -m tad.cli --device {DEVICE} run-e1 --config {CONFIG}

### (Optional) run stages individually instead of `run-e1`
```bash
python -m tad.cli --device cuda generate-trajectories --config <cfg>
python -m tad.cli            build-dataset          --config <cfg>
python -m tad.cli --device cuda train-predictor     --config <cfg>
python -m tad.cli --device cuda run-online-optimizer --config <cfg>
python -m tad.cli            make-report            --experiment-dir artifacts/experiments/<name>
```

In [ ]:
# 6. Show the generated report (Markdown + figures)
import yaml, glob, os
from IPython.display import Markdown, Image, display
name = yaml.safe_load(open(CONFIG))['experiment']['name']
exp_dir = os.path.join('artifacts', 'experiments', name)
report_md = os.path.join(exp_dir, 'report', 'report.md')
display(Markdown(open(report_md).read()))
for png in sorted(glob.glob(os.path.join(exp_dir, 'report', '*.png'))):
    print(png)
    display(Image(png))

In [ ]:
# 7. (Optional) zip the experiment outputs for download
import shutil, os
archive = shutil.make_archive(name, 'zip', exp_dir)
print('wrote', archive)
try:
    from google.colab import files
    files.download(archive)
except Exception as e:
    print('manual download:', archive, '(', e, ')')